# 2. CART Phenotyping

Classification and Regression Tree (CART) analysis to define TNM-based prognostic phenotypes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load prepared cohort
df = pd.read_csv('LiverMets_Final_Dataset.csv')

# Apply inclusion criteria (from notebook 1)
complete_tnm = df[
    (df['T_STAGE'].notna()) & (df['T_STAGE'] != 'ND') &
    (df['N_STAGE'].notna()) & (df['N_STAGE'] != 'ND') &
    (df['M_STAGE'].notna()) & (df['M_STAGE'] != 'ND')
]
included = complete_tnm[
    (complete_tnm['SURVIVAL_YEARS'].notna()) & 
    (complete_tnm['SURVIVAL_YEARS'] > 0) &
    (complete_tnm['VITAL_STATUS'].notna())
]
print(f"Cohort: {len(included):,} patients")

## Define CART Phenotypes

Phenotypes based on TNM binary classification tree:
- **Phenotype 1 (Favourable)**: M0, N0–N1
- **Phenotype 2 (Intermediate)**: (M0, N2) OR (M1, N0–N1)
- **Phenotype 3 (Adverse)**: M1, N2

In [ ]:
# Define phenotypes
df_ph = included.copy()
df_ph['PHENOTYPE'] = np.nan

# Phenotype 1: M0, N0-N1
mask1 = (df_ph['M_STAGE'] == 'M0') & (df_ph['N_STAGE'].isin(['N0', 'N1']))
df_ph.loc[mask1, 'PHENOTYPE'] = 1

# Phenotype 2: (M0, N2) or (M1, N0-N1)
mask2a = (df_ph['M_STAGE'] == 'M0') & (df_ph['N_STAGE'] == 'N2')
mask2b = (df_ph['M_STAGE'] == 'M1') & (df_ph['N_STAGE'].isin(['N0', 'N1']))
df_ph.loc[mask2a | mask2b, 'PHENOTYPE'] = 2

# Phenotype 3: M1, N2
mask3 = (df_ph['M_STAGE'] == 'M1') & (df_ph['N_STAGE'] == 'N2')
df_ph.loc[mask3, 'PHENOTYPE'] = 3

# Count patients in each phenotype
phenotype_counts = df_ph['PHENOTYPE'].value_counts().sort_index()
print("Phenotype Distribution:")
for ph, count in phenotype_counts.items():
    pct = 100 * count / len(df_ph)
    print(f"  Phenotype {int(ph)}: {count:,} ({pct:.1f}%)")
print(f"  Total: {len(df_ph):,}")

## Phenotype Characteristics

In [ ]:
# Baseline characteristics by phenotype
for ph in [1, 2, 3]:
    cohort = df_ph[df_ph['PHENOTYPE'] == ph]
    print(f"\n{'='*60}")
    print(f"PHENOTYPE {int(ph)} (n={len(cohort):,})")
    print(f"{'='*60}")
    
    # TNM characteristics
    print(f"\nTNM:")
    print(f"  T-stage: {cohort['T_STAGE'].value_counts().to_dict()}")
    print(f"  N-stage: {cohort['N_STAGE'].value_counts().to_dict()}")
    print(f"  M-stage: {cohort['M_STAGE'].value_counts().to_dict()}")
    
    # Demographics
    age_mean = cohort['AGE_AT_REFERRAL'].mean()
    age_std = cohort['AGE_AT_REFERRAL'].std()
    male_pct = 100 * (cohort['GENDER'].str.upper() == 'MALE').sum() / len(cohort)
    print(f"\nDemographics:")
    print(f"  Age: {age_mean:.1f} ± {age_std:.1f} years")
    print(f"  Male: {male_pct:.1f}%")
    
    # Treatment
    print(f"\nTreatment:")
    for treat in cohort['TREATMENT'].value_counts().index:
        count = (cohort['TREATMENT'] == treat).sum()
        pct = 100 * count / len(cohort)
        print(f"  {treat}: {count:,} ({pct:.1f}%)")
    
    # Outcomes
    deaths = (cohort['VITAL_STATUS'] == 1).sum()
    print(f"\nOutcomes:")
    print(f"  Deaths: {deaths:,} ({100*deaths/len(cohort):.1f}%)")
    print(f"  Follow-up: {cohort['SURVIVAL_YEARS'].mean():.2f} years")

## Phenotype Distribution Plot

In [ ]:
# Plot phenotype distribution
fig, ax = plt.subplots(figsize=(10, 6))

phenotypes = ['Favourable\n(M0, N0–N1)', 'Intermediate\n(M0 N2 or M1 N0–N1)', 'Adverse\n(M1, N2)']
counts = [phenotype_counts[1], phenotype_counts[2], phenotype_counts[3]]

bars = ax.bar(phenotypes, counts, color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='black', linewidth=1.5)

# Add count labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{int(height):,}',
           ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Number of Patients', fontsize=12, fontweight='bold')
ax.set_ylim([0, max(counts) * 1.15])
ax.grid(True, alpha=0.3, axis='y')
ax.set_title('CART Phenotype Distribution (n=14,759)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()